In [ ]:
# --- core ---
import numpy as np
import xarray as xr
import pandas as pd

# --- plotting ---
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- mapping (cartopy) ---
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

# --- misc helpers ---
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------
# Open the ERA5 NetCDF file
# ----------------------------
PATH = Path("../data/era5_2021-nov_250-500-925_uv_pv_gph.nc")

ds = xr.open_dataset(PATH)  # add chunks=... later if you want dask
display(ds)

print("Variables:", list(ds.data_vars))
print("Coords:", list(ds.coords))
for k, v in ds.dims.items():
    print(f"dim {k}: {v}")

In [ ]:
ds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

t = np.datetime64("2021-11-12T03:00:00")
snap = ds.sel(valid_time=t, method="nearest")

lats = snap["latitude"].values
lons = snap["longitude"].values  # 0..360
levels = snap["pressure_level"].values

proj = ccrs.PlateCarree()

fig, axes = plt.subplots(
    nrows=1, ncols=len(levels),
    figsize=(18, 5),
    subplot_kw={"projection": proj},
    constrained_layout=True
)

if len(levels) == 1:
    axes = [axes]

for ax, p in zip(axes, levels):
    pv = snap["pv"].sel(pressure_level=p)

    pv_cyc, lons_cyc = add_cyclic_point(pv.values, coord=lons)

    # independent normalization per panel
    vmin = np.nanmin(pv_cyc)
    vmax = np.nanmax(pv_cyc)

    im = ax.pcolormesh(
        lons_cyc, lats, pv_cyc,
        transform=proj,
        shading="auto",
        vmin=vmin,
        vmax=vmax
    )

    ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="black", linewidth=0.6)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_global()

    ax.set_title(f"PV @ {float(p):g} hPa\n{str(snap['valid_time'].values)[:19]}")

    # colorbar per panel (independent scale)
    cbar = fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.85, pad=0.02)
    cbar.set_label("Potential vorticity (file units)")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

t = np.datetime64("2021-11-12T03:00:00")
snap = ds.sel(valid_time=t, method="nearest")

lats = snap["latitude"].values
lons = snap["longitude"].values  # 0..360
levels = snap["pressure_level"].values

g0 = 9.80665  # m/s^2

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    nrows=1, ncols=len(levels),
    figsize=(18, 5),
    subplot_kw={"projection": proj},
    constrained_layout=True
)
if len(levels) == 1:
    axes = [axes]

# stride trades speed vs detail: 2 is decent, 3-4 is much faster
STRIDE = 2

for ax, p in zip(axes, levels):
    z = snap["z"].sel(pressure_level=p).values  # (lat, lon)

    # convert geopotential -> geopotential height (m)
    z_m = z / g0

    # add cyclic longitude to avoid seam at 0/360
    z_cyc, lons_cyc = add_cyclic_point(z_m, coord=lons)

    # decimate for speed
    z_plot = z_cyc[::STRIDE, ::STRIDE]
    lats_plot = lats[::STRIDE]
    lons_plot = lons_cyc[::STRIDE]

    vmin = float(np.nanmin(z_plot))
    vmax = float(np.nanmax(z_plot))

    # "standard-ish" contour intervals by level
    if float(p) == 500.0:
        step = 60.0
    elif float(p) == 250.0:
        step = 120.0
    else:  # 925
        step = 30.0

    clevs = np.arange(np.floor(vmin/step)*step, np.ceil(vmax/step)*step + step, step)

    cs = ax.contour(
        lons_plot,
        lats_plot,
        z_plot,
        levels=clevs,
        colors="k",          # <-- high contrast
        linewidths=0.8,
        transform=proj,
    )

    # labels are expensive; fewer labels = faster
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.0f")

    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.set_global()
    ax.set_title(f"GPH contours (m) @ {float(p):g} hPa\n{str(snap['valid_time'].values)[:19]}")

plt.show()